# AIST-FYP Colab Evaluation Notebook

This notebook runs **RAGTruth/CiteEval evaluation** and **verifier module evaluation** using the repository codebase.

## What this notebook does
1. Mounts Google Drive and clones the repo
2. Installs project dependencies
3. Validates prebuilt retrieval artifacts and benchmark data
4. Runs baseline RAGTruth evaluation
5. Runs verifier module evaluation (RAGTruth and optional CiteBench)
6. Runs standard CiteEval evaluation (system and optional metric track)
7. Saves outputs back to Drive

## 📂 Artifact Placement Checklist

Before running the evaluation, ensure your repository has the necessary artifacts in the following structure:

- `data/`
  - `indexes/`
    - `{STRATEGY}/` (e.g., `production/`)
      - `faiss.index`
      - `metadata.pkl`
  - `processed/`
    - `wiki_chunks_{STRATEGY}.jsonl`
- `benchmark/`
  - `RAGTruth/`
    - `dataset/`
  - `CiteEval/`

If these are not in the repository you clone, you must upload them manually to the `/content/AIST-FYP/` directory after cloning.

## 🔑 Setup API Keys (Colab Secrets)

To run the evaluation, you likely need API keys for the LLM providers (DeepSeek, OpenAI, etc.). Colab provides a secure way to store these using the **Secrets** feature.

1. Click on the **Key icon** (Secrets) in the left sidebar.
2. Add a new secret with the name:
   - `OPENAI_API_KEY`: Your OpenAI API key.
   - `DEEPSEEK_API_KEY`: Your DeepSeek API key.
   - `HUGGINGFACE_TOKEN`: (Optional) For gated models.
3. Make sure to toggle the **"Notebook access"** switch to **ON** for this notebook.

The next cell will automatically load these secrets into the environment variables used by the project scripts.

In [ ]:
# ==============================
# Setup API Keys (Secrets)
# ==============================
try:
    from google.colab import userdata
    import os

    # Define keys to load
    secret_keys = ["OPENAI_API_KEY", "DEEPSEEK_API_KEY", "HUGGINGFACE_TOKEN"]
    loaded_any = False

    for key in secret_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
                print(f"✅ Loaded secret: {key}")
                loaded_any = True
        except Exception:
            # Silently skip if not found or no access
            pass
    
    if not loaded_any:
        print("ℹ️ No secrets loaded. If you need API keys, add them via the 🔑 (Secrets) tab.")
except ImportError:
    print("⚠️ 'google.colab.userdata' not found. If running locally, please export your API keys manually.")

### Configuration (Smoke Test Settings)
This notebook defaults to a **smoke test** for both RAGTruth and CiteEval datasets. 

**For full evaluation:**
1. In the cell below, change `RAGTRUTH_MAX_SAMPLES = 10` and `CITEEVAL_MAX_SAMPLES = 10` to `None`.
2. Run the notebook.

In [ ]:
# ==============================
# Parameters (edit this cell)
# ==============================
REPO_URL = "https://github.com/xiashuidaolaoshuren/AIST-FYP.git"
REPO_BRANCH = "main"
REPO_DIR = "/content/AIST-FYP"
COLAB_ENV_PROJECT = "colab/env"
COLAB_UV_EXTRAS = ["evaluation"]
INSTALL_SPACY_MODEL = True

RUN_RAGTRUTH = True
RUN_CITEEVAL = True
RUN_CITEEVAL_METRIC = False  # requires metric data + human labels

# Verifier module evaluation switches
RUN_RAGTRUTH_VERIFIER_MODULE_EVAL = True
RUN_CITEEVAL_VERIFIER_MODULE_EVAL = False
VERIFIER_VARIANTS = [
    "baseline",
    "verifier_intrinsic_only",
    "verifier_grounded_only",
    "verifier_nli_only",
    "verifier_self_agreement_only",
]

# Evaluation knobs
RAGTRUTH_SPLIT = "test"
RAGTRUTH_MAX_SAMPLES = 100   # None for full split
RAGTRUTH_BATCH_SIZE = 10
RAGTRUTH_EVAL_MODE = "ragtruth_eval"  # ragtruth_eval | normal | gold_context_generation
STRATEGY = "validation"      # development | validation | production

# Qwen 2.5 colab profile knobs
GENERATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"
GENERATOR_MAX_INPUT_TOKENS = 4096
GENERATOR_LOAD_IN_8BIT = True

# CiteEval CLI knobs (direct script call)
CITEEVAL_PROVIDER = "deepseek"       # deepseek | openai
CITEEVAL_MODEL_NAME = ""             # empty => script default by provider
CITEEVAL_CONTEXT_SOURCE = "retrieval"  # retrieval | oracle (system track label)
CITEEVAL_MAX_EXAMPLES = 10
CITEEVAL_MAX_SAMPLES = 10
CITEEVAL_METRIC_SPLIT = "test"       # dev | test
CITEEVAL_DRY_RUN_FIRST = True         # run --dry-run before actual execution

# ⚠️ REMINDER: Ensure the following artifacts are placed in the repo correctly:
# 1. FAISS Indexes: {REPO_DIR}/data/indexes/{STRATEGY}/faiss.index
# 2. Index Metadata: {REPO_DIR}/data/indexes/{STRATEGY}/metadata.pkl
# 3. Wikipedia Chunks: {REPO_DIR}/data/processed/wiki_chunks_{STRATEGY}.jsonl
# 4. Benchmarks: {REPO_DIR}/benchmark/RAGTruth/dataset/ and {REPO_DIR}/benchmark/CiteEval/
ARTIFACTS_IN_PROJECT = True
DRIVE_DATA_ROOT = "/content/drive/MyDrive/data"
DRIVE_RAGTRUTH_DATASET_ROOT = "/content/drive/MyDrive/AIST-FYP/benchmark/RAGTruth/dataset"
DRIVE_CITEEVAL_ROOT = "/content/drive/MyDrive/AIST-FYP/benchmark/CiteEval"

# Persistent storage policy (survive Colab runtime resets)
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/AIST-FYP-colab-outputs"
DRIVE_WORK_ROOT = f"{DRIVE_OUTPUT_DIR}/work_eval"
LOCAL_WORK_ROOT = DRIVE_WORK_ROOT
RUN_TAG = "colab_evaluation"
SKIP_EXPORT_COPY_WHEN_PERSISTENT = True

# Persistent runtime output dirs
RAGTRUTH_OUTPUT_DIR = f"{LOCAL_WORK_ROOT}/outputs/ragtruth_eval"
VERIFIER_RAG_OUTPUT_DIR = f"{LOCAL_WORK_ROOT}/outputs/verifier_eval_ragtruth"
VERIFIER_CITE_OUTPUT_DIR = f"{LOCAL_WORK_ROOT}/outputs/verifier_eval_citebench"
CITEEVAL_SYSTEM_OUTPUTS_DIR = f"{LOCAL_WORK_ROOT}/citeeval/system_eval_outputs"
CITEEVAL_METRIC_OUTPUTS_DIR = f"{LOCAL_WORK_ROOT}/citeeval/metric_eval_outputs"
CITEEVAL_TMP_SAMPLING_DIR = f"{LOCAL_WORK_ROOT}/citeeval/tmp/sampling"

In [ ]:
import os
import json
import shutil
import subprocess
import sys
from pathlib import Path
from datetime import datetime

def _normalized_unique_paths(entries):
    ordered = []
    seen = set()
    for entry in entries:
        if not entry:
            continue
        resolved = str(Path(entry).resolve())
        if resolved not in seen:
            ordered.append(resolved)
            seen.add(resolved)
    return ordered

def run(cmd, cwd=None, check=True, stream=True):
    env = os.environ.copy()
    repo_path = str(Path(cwd).resolve()) if cwd else str(Path(os.getcwd()).resolve())

    existing_pp = env.get('PYTHONPATH', '').split(os.pathsep)
    pp_entries = _normalized_unique_paths([repo_path, *existing_pp])

    site_pkgs = [p for p in sys.path if 'site-packages' in p]
    pp_entries = _normalized_unique_paths([*pp_entries, *site_pkgs])

    env['PYTHONPATH'] = os.pathsep.join(pp_entries)

    print(f"\n$ {cmd}")

    if stream:
        process = subprocess.Popen(
            cmd,
            shell=True,
            cwd=cwd,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            bufsize=1,
            executable='/bin/bash',
        )
        out_lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            out_lines.append(line)
        process.wait()
        completed = subprocess.CompletedProcess(
            args=cmd,
            returncode=process.returncode,
            stdout="".join(out_lines),
            stderr=None,
        )
    else:
        completed = subprocess.run(
            cmd,
            shell=True,
            cwd=cwd,
            env=env,
            text=True,
            capture_output=True,
            executable='/bin/bash',
        )
        if completed.stdout:
            print(completed.stdout)

    if completed.returncode != 0:
        if not stream and completed.stderr:
            print(completed.stderr)
        if check:
            raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def ensure_symlink_dir(link_path: Path, target_path: Path):
    target_path = Path(target_path)
    link_path = Path(link_path)
    ensure_dir(target_path)
    ensure_dir(link_path.parent)

    if link_path.is_symlink():
        current_target = Path(os.readlink(link_path))
        if current_target == target_path:
            return
        link_path.unlink()
    elif link_path.exists():
        if link_path.is_dir():
            shutil.rmtree(link_path)
        else:
            link_path.unlink()

    os.symlink(target_path, link_path, target_is_directory=True)

def copytree_merge(src, dst):
    src_p = Path(src)
    dst_p = Path(dst)
    if not src_p.exists():
        return
    for p in src_p.rglob('*'):
        rel = p.relative_to(src_p)
        t = dst_p / rel
        if p.is_dir():
            t.mkdir(parents=True, exist_ok=True)
        else:
            t.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, t)

def exists_or_raise(path, msg):
    if not Path(path).exists():
        raise FileNotFoundError(f"{msg}: {path}")

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

if not str(LOCAL_WORK_ROOT).startswith('/content/drive/'):
    print(f"⚠️ LOCAL_WORK_ROOT is not on Drive: {LOCAL_WORK_ROOT}")
    print("Artifacts may not survive Colab runtime reset.")

ensure_dir(LOCAL_WORK_ROOT)
print("Persistent LOCAL_WORK_ROOT:", LOCAL_WORK_ROOT)

# Clone/update repo
if Path(REPO_DIR).exists() and (Path(REPO_DIR) / '.git').exists():
    print(f"Repo dir already exists: {REPO_DIR}")
    run("git fetch --all", cwd=REPO_DIR, check=False)
    run(f"git checkout {REPO_BRANCH}", cwd=REPO_DIR, check=False)
    run(f"git pull origin {REPO_BRANCH}", cwd=REPO_DIR, check=False)
else:
    run(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

run("git rev-parse --abbrev-ref HEAD", cwd=REPO_DIR)
run("git log -1 --oneline", cwd=REPO_DIR)

# Ensure repo root is available first for script imports (src.*).
repo_root = str(Path(REPO_DIR))
existing_pp = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
ordered_pp = []
for entry in [repo_root, *existing_pp]:
    if entry not in ordered_pp:
        ordered_pp.append(entry)
os.environ['PYTHONPATH'] = os.pathsep.join(ordered_pp)
print('PYTHONPATH (repo-first):', os.environ['PYTHONPATH'])
print('src/utils/config.py exists:', (Path(REPO_DIR) / 'src' / 'utils' / 'config.py').exists())

# Link project data path to Drive data artifacts root
drive_data_root = Path(DRIVE_DATA_ROOT)
if not drive_data_root.exists():
    raise FileNotFoundError(
        f"Drive data root not found: {drive_data_root}. Place artifacts under this path before running evaluation."
    )

project_data_path = Path(REPO_DIR) / 'data'
ensure_symlink_dir(project_data_path, drive_data_root)
resolved_data_path = project_data_path.resolve()
print(f"Data artifact path: {project_data_path} -> {resolved_data_path}")

# Link RAGTruth dataset path to Drive if available
drive_ragtruth_dataset = Path(DRIVE_RAGTRUTH_DATASET_ROOT)
project_ragtruth_dataset = Path(REPO_DIR) / 'benchmark' / 'RAGTruth' / 'dataset'
if drive_ragtruth_dataset.exists():
    ensure_symlink_dir(project_ragtruth_dataset, drive_ragtruth_dataset)
    resolved_ragtruth_dataset = project_ragtruth_dataset.resolve()
    print(f"RAGTruth dataset path: {project_ragtruth_dataset} -> {resolved_ragtruth_dataset}")
elif RUN_RAGTRUTH:
    print(
        f"⚠️ Drive RAGTruth dataset not found at {drive_ragtruth_dataset}. "
        "If your dataset is elsewhere, update DRIVE_RAGTRUTH_DATASET_ROOT in the parameters cell."
    )

# Link CiteEval/CiteBench benchmark folder to Drive if available
drive_citeeval_root = Path(DRIVE_CITEEVAL_ROOT)
project_citeeval_root = Path(REPO_DIR) / 'benchmark' / 'CiteEval'
if drive_citeeval_root.exists():
    ensure_symlink_dir(project_citeeval_root, drive_citeeval_root)
    resolved_citeeval_root = project_citeeval_root.resolve()
    print(f"CiteEval benchmark path: {project_citeeval_root} -> {resolved_citeeval_root}")
elif RUN_CITEEVAL or RUN_CITEEVAL_METRIC or RUN_CITEEVAL_VERIFIER_MODULE_EVAL:
    print(
        f"⚠️ Drive CiteEval root not found at {drive_citeeval_root}. "
        "If your benchmark is elsewhere, update DRIVE_CITEEVAL_ROOT in the parameters cell."
    )

# Persist runtime outputs via symlink to Drive-backed workspace
symlink_map = {
    Path(REPO_DIR) / 'outputs' / 'ragtruth_eval': Path(RAGTRUTH_OUTPUT_DIR),
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval': Path(VERIFIER_RAG_OUTPUT_DIR),
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval_citebench': Path(VERIFIER_CITE_OUTPUT_DIR),
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'system_eval_outputs': Path(CITEEVAL_SYSTEM_OUTPUTS_DIR),
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'metric_eval_outputs': Path(CITEEVAL_METRIC_OUTPUTS_DIR),
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'tmp' / 'sampling': Path(CITEEVAL_TMP_SAMPLING_DIR),
}

for link_path, target_path in symlink_map.items():
    ensure_symlink_dir(link_path, target_path)
    print(f"Persisted path: {link_path} -> {target_path}")

In [ ]:
# Install dependencies
run("python -m pip install -U pip wheel setuptools", stream=True)
run("python -m pip install -U faiss-cpu rank_bm25", stream=True)
run("python -m pip install -U uv", stream=True)

uv_project = Path(REPO_DIR) / COLAB_ENV_PROJECT
extras_args = " ".join(f"--extra {extra}" for extra in COLAB_UV_EXTRAS)
sync_cmd = f"uv sync --project {uv_project} {extras_args}"
result = run(sync_cmd, cwd=REPO_DIR, check=False, stream=True)

spacy_model_wheel = "https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl"
spacy_install_cmd = "python -m spacy download en_core_web_sm"

if result.returncode == 0:
    uv_python = uv_project / ".venv" / "bin" / "python"
    os.environ["PATH"] = f"{uv_python.parent}:{os.environ.get('PATH', '')}"
    spacy_install_cmd = f"uv pip install --python {uv_python} {spacy_model_wheel}"
    run(f"{uv_python} - <<\"PY\"\nimport sys\nprint('python executable:', sys.executable)\nPY", cwd=REPO_DIR)
    print(f"✅ uv sync complete: {uv_project}")
else:
    print('\n⚠️ uv sync failed. Falling back to pip requirements install...')
    requirements_path = Path(REPO_DIR) / 'requirements.txt'
    pytorch_index = 'https://download.pytorch.org/whl/cu121'
    install_cmd = f"pip install --extra-index-url {pytorch_index} -r {requirements_path}"
    fallback_result = run(install_cmd, cwd=REPO_DIR, check=False, stream=True)

    if fallback_result.returncode != 0:
        print('\n⚠️ Full requirements install failed. Falling back to Colab-torch-compatible install...')
        filtered = []
        skip_prefixes = ('torch==', 'torchvision==', 'torchaudio==')
        for raw in requirements_path.read_text(encoding='utf-8').splitlines():
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            if any(line.startswith(prefix) for prefix in skip_prefixes):
                continue
            filtered.append(line)

        temp_req = Path(REPO_DIR) / 'requirements.colab.filtered.txt'
        temp_req.write_text('\n'.join(filtered) + '\n', encoding='utf-8')
        run(f"pip install -r {temp_req}", cwd=REPO_DIR, stream=True)

# spaCy model required by verifier
if INSTALL_SPACY_MODEL:
    run(spacy_install_cmd, cwd=REPO_DIR, stream=True)

In [ ]:
# Runtime and API key setup
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Set keys in Colab before running CiteEval modules that need them:
# os.environ['DEEPSEEK_API_KEY'] = '...'
# os.environ['OPENAI_API_KEY'] = '...'

# Provider defaults from parameter cell
os.environ['CITEEVAL_PROVIDER'] = CITEEVAL_PROVIDER
os.environ['CITEEVAL_ROOT'] = str(Path(REPO_DIR) / 'benchmark/CiteEval')

repo_root = str(Path(REPO_DIR).resolve())

# Critical: keep only repo root in global PYTHONPATH to avoid src-package shadowing.
# CiteEval scripts are executed from repo context and should not globally override src.*
existing_pp = [p for p in os.environ.get('PYTHONPATH', '').split(os.pathsep) if p]
ordered_pp = []
seen = set()
for entry in [repo_root, *existing_pp]:
    resolved = str(Path(entry).resolve())
    if resolved not in seen:
        ordered_pp.append(resolved)
        seen.add(resolved)
os.environ['PYTHONPATH'] = os.pathsep.join(ordered_pp)

print('PYTHONPATH (top 6):', ordered_pp[:6])
print('CITEEVAL_PROVIDER =', os.environ.get('CITEEVAL_PROVIDER'))
print('CITEEVAL_MODEL_NAME =', CITEEVAL_MODEL_NAME if CITEEVAL_MODEL_NAME else '(script default)')
print('DEEPSEEK_API_KEY set =', bool(os.environ.get('DEEPSEEK_API_KEY')))
print('OPENAI_API_KEY set =', bool(os.environ.get('OPENAI_API_KEY')))

if os.environ['PYTHONPATH'].split(os.pathsep)[0] != repo_root:
    raise RuntimeError('PYTHONPATH ordering error: repo root is not first entry.')

In [ ]:
# Artifacts check: using project-local folder structure (no external hydration)
if ARTIFACTS_IN_PROJECT:
    data_root = Path(REPO_DIR) / 'data'
    benchmark_root = Path(REPO_DIR) / 'benchmark'
    ragtruth_dataset_root = benchmark_root / 'RAGTruth' / 'dataset'
    required_roots = [data_root, benchmark_root]
    for root in required_roots:
        if root.exists():
            print('Found:', root)
        else:
            print('Missing expected folder:', root)

    # Sanity-check that project data resolves to the Drive artifacts root.
    resolved_data_root = data_root.resolve()
    expected_data_root = Path(DRIVE_DATA_ROOT).resolve()
    print('Resolved data root:', resolved_data_root)
    print('Expected data root:', expected_data_root)
    if resolved_data_root != expected_data_root:
        raise RuntimeError(
            f"Data root mismatch: {resolved_data_root} != {expected_data_root}. Re-run setup cell to relink data path."
        )

    if RUN_RAGTRUTH and ragtruth_dataset_root.exists():
        print('Resolved RAGTruth dataset root:', ragtruth_dataset_root.resolve())
else:
    print('ARTIFACTS_IN_PROJECT=False (no copy step configured).')

In [ ]:
# Validate required paths for selected strategy
repo = Path(REPO_DIR)
faiss_index = repo / f"data/indexes/{STRATEGY}/faiss.index"
index_meta = repo / f"data/indexes/{STRATEGY}/metadata.pkl"
chunks_file = repo / f"data/processed/wiki_chunks_{STRATEGY}.jsonl"
ragtruth_dataset = repo / 'benchmark/RAGTruth/dataset'
citeeval_root = repo / 'benchmark/CiteEval'

exists_or_raise(faiss_index, 'Missing FAISS index')
exists_or_raise(index_meta, 'Missing index metadata')
exists_or_raise(chunks_file, 'Missing processed chunks')

if RUN_RAGTRUTH:
    exists_or_raise(ragtruth_dataset, 'Missing RAGTruth dataset directory')

if RUN_CITEEVAL or RUN_CITEEVAL_METRIC or RUN_CITEEVAL_VERIFIER_MODULE_EVAL:
    exists_or_raise(citeeval_root, 'Missing CiteEval benchmark directory')

print('Preflight path checks passed.')

In [ ]:
# Import-path sanity check for src package
import sys
from pathlib import Path

repo = Path(REPO_DIR).resolve()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.utils.config import Config  # noqa: F401
print('Import check passed: src.utils.config')

In [ ]:
# Create a Colab-specific config file based on config.yaml
import yaml

base_config = Path(REPO_DIR) / 'config.yaml'
colab_config = Path(REPO_DIR) / 'config.colab.yaml'

with open(base_config, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Keep GPU for model inference; force CPU for FAISS runtime
cfg['processing']['device'] = 'cuda'
cfg['verification']['nli']['device'] = 'cuda'
cfg['verification']['self_agreement']['device'] = 'cuda'
cfg.setdefault('retrieval', {}).setdefault('faiss', {})['use_gpu'] = False
cfg['retrieval']['faiss']['gpu_id'] = 0

# Qwen 2.5 generation profile
cfg.setdefault('models', {})['generator'] = GENERATOR_MODEL
cfg.setdefault('generation', {})['max_input_tokens'] = int(GENERATOR_MAX_INPUT_TOKENS)
cfg['generation']['load_in_8bit'] = bool(GENERATOR_LOAD_IN_8BIT)

# Ensure evaluation mode defaults
cfg.setdefault('processing', {}).setdefault('query_split', {})['enabled'] = False
cfg.setdefault('evaluation', {}).setdefault('benchmarks', {}).setdefault('ragtruth', {})['ragtruth_eval_mode'] = RAGTRUTH_EVAL_MODE

with open(colab_config, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print('Wrote', colab_config)
print('FAISS runtime in Colab config: CPU (retrieval.faiss.use_gpu=False)')
print('Generator model:', cfg['models']['generator'])
print('Max input tokens:', cfg['generation']['max_input_tokens'])
print('Load in 8-bit:', cfg['generation']['load_in_8bit'])

In [ ]:
# Validate Qwen config and tokenizer budget
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
print('Tokenizer model_max_length:', tok.model_max_length)
if torch.cuda.is_available():
    total_mb = torch.cuda.get_device_properties(0).total_memory // (1024 * 1024)
    reserved_mb = torch.cuda.memory_reserved(0) // (1024 * 1024)
    print(f'CUDA memory total/reserved MB: {total_mb}/{reserved_mb}')
print('Configured max_input_tokens:', GENERATOR_MAX_INPUT_TOKENS)

In [ ]:
# Run RAGTruth evaluation (direct script call)
import sys
from pathlib import Path

if RUN_RAGTRUTH:
    repo = Path(REPO_DIR).resolve()
    max_samples_arg = '' if RAGTRUTH_MAX_SAMPLES is None else f" --max-samples {RAGTRUTH_MAX_SAMPLES}"
    python_exec = sys.executable

    config_file = repo / 'config.colab.yaml'
    if not config_file.exists():
        print(f"⚠️ Warning: {config_file} not found. Using default config.yaml.")
        config_arg = '--config config.yaml'
    else:
        config_arg = '--config config.colab.yaml'

    cmd = (
        f'"{python_exec}" scripts/demo_ragtruth_eval.py {config_arg} --split {RAGTRUTH_SPLIT} '
        f"--batch-size {RAGTRUTH_BATCH_SIZE} --strategy {STRATEGY} --save-results "
        f"--ragtruth-eval-mode {RAGTRUTH_EVAL_MODE}{max_samples_arg}"
    )

    print('Running command with repo-first PYTHONPATH...')
    run(cmd, cwd=str(repo))
else:
    print('RUN_RAGTRUTH=False, skipped.')

In [ ]:
# Summarize latest RAGTruth result
ragtruth_out = Path(REPO_DIR) / 'outputs/ragtruth_eval'
if ragtruth_out.exists():
    files = sorted(ragtruth_out.glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    if files:
        latest = files[0]
        print('Latest RAGTruth file:', latest)
        payload = json.loads(latest.read_text(encoding='utf-8'))
        metrics = payload.get('metrics', {}).get('overall', payload.get('overall', {}))
        if metrics:
            print({k: metrics.get(k) for k in ['num_samples', 'accuracy', 'precision', 'recall', 'f1']})

        ensure_dir(DRIVE_OUTPUT_DIR)
        shutil.copy2(latest, Path(DRIVE_OUTPUT_DIR) / latest.name)
        print('Copied to Drive:', Path(DRIVE_OUTPUT_DIR) / latest.name)
    else:
        print('No RAGTruth JSON found.')
else:
    print('No outputs/ragtruth_eval directory found.')

In [ ]:
# Run verifier module evaluation on RAGTruth
import sys

if RUN_RAGTRUTH_VERIFIER_MODULE_EVAL:
    verifier_variants_arg = " ".join(VERIFIER_VARIANTS)
    max_samples_arg = "" if RAGTRUTH_MAX_SAMPLES is None else f" --max-samples {RAGTRUTH_MAX_SAMPLES}"
    python_exec = sys.executable
    cmd = (
        f'"{python_exec}" scripts/evaluate_mitigation_strategy.py '
        "--config config.colab.yaml "
        f"--split {RAGTRUTH_SPLIT} "
        f"--batch-size {RAGTRUTH_BATCH_SIZE} "
        f"--strategy {STRATEGY} "
        f"--ragtruth-eval-mode {RAGTRUTH_EVAL_MODE} "
        f"--variants {verifier_variants_arg}"
        f"{max_samples_arg}"
    )
    run(cmd, cwd=REPO_DIR)
else:
    print("RUN_RAGTRUTH_VERIFIER_MODULE_EVAL=False, skipped.")

In [ ]:
# Prepare CiteEval system input + preflight checks (direct script call)
if RUN_CITEEVAL:
    if SYSTEM_INPUT_JSON:
        cmd = (
            f"python scripts/convert_to_citeeval.py --input {SYSTEM_INPUT_JSON} "
            f"--output benchmark/CiteEval/data/system_eval/my_pipeline_results.json --strategy {STRATEGY}"
        )
        run(cmd, cwd=REPO_DIR)
    else:
        print('SYSTEM_INPUT_JSON is empty. Using an existing system_eval JSON if present.')

    provider = CITEEVAL_PROVIDER.strip().lower()
    if provider == 'deepseek' and not os.environ.get('DEEPSEEK_API_KEY'):
        raise EnvironmentError('DEEPSEEK_API_KEY is required when CITEEVAL_PROVIDER=deepseek')
    if provider == 'openai' and not os.environ.get('OPENAI_API_KEY'):
        raise EnvironmentError('OPENAI_API_KEY is required when CITEEVAL_PROVIDER=openai')

    system_eval_dir = Path(REPO_DIR) / 'benchmark/CiteEval/data/system_eval'
    system_jsons = sorted(system_eval_dir.glob('*.json')) if system_eval_dir.exists() else []
    if not system_jsons:
        raise FileNotFoundError('No system_eval JSON found for CiteEval system track.')
    print('System eval input candidates:', [p.name for p in system_jsons][:5])

    # Optional dry-run to validate command wiring before real run
    if CITEEVAL_DRY_RUN_FIRST:
        dry_run_cmd = (
            'python scripts/run_citebench_eval.py --evaluation-role both --track both '
            f'--context-source {CITEEVAL_CONTEXT_SOURCE} --dry-run'
        )
        run(dry_run_cmd, cwd=REPO_DIR)
else:
    print('RUN_CITEEVAL=False, skipped.')

In [ ]:
# Run CiteEval (direct script call; system track always, metric optional)
if RUN_CITEEVAL:
    system_eval_dir = Path(REPO_DIR) / 'benchmark/CiteEval/data/system_eval'
    system_jsons = sorted(system_eval_dir.glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True)
    system_input = system_jsons[0]

    provider_arg = f" --provider {CITEEVAL_PROVIDER}" if CITEEVAL_PROVIDER else ''
    model_arg = f" --model-name {CITEEVAL_MODEL_NAME}" if CITEEVAL_MODEL_NAME else ''
    max_examples_arg = '' if CITEEVAL_MAX_EXAMPLES is None else f" --max-examples {CITEEVAL_MAX_EXAMPLES}"

    system_cmd = (
        "python scripts/run_citebench_eval.py "
        "--evaluation-role mitigation --track system "
        f"--system-input {system_input} "
        f"--context-source {CITEEVAL_CONTEXT_SOURCE}"
        f"{provider_arg}{model_arg}{max_examples_arg}"
    )
    run(system_cmd, cwd=REPO_DIR)

    if RUN_CITEEVAL_METRIC:
        metric_cmd = (
            "python scripts/run_citebench_eval.py "
            "--evaluation-role baseline --track metric "
            f"--metric-split {CITEEVAL_METRIC_SPLIT}"
            f"{provider_arg}{model_arg}{max_examples_arg}"
        )
        run(metric_cmd, cwd=REPO_DIR)
else:
    print('RUN_CITEEVAL=False, skipped.')

In [ ]:
# Run verifier module evaluation on CiteBench (optional)
if RUN_CITEEVAL_VERIFIER_MODULE_EVAL:
    verifier_variants_arg = " ".join(VERIFIER_VARIANTS)
    provider = CITEEVAL_PROVIDER.strip().lower()
    model_name = CITEEVAL_MODEL_NAME.strip() if CITEEVAL_MODEL_NAME else ("deepseek-chat" if provider == "deepseek" else "gpt-4o")
    cmd = (
        "python scripts/evaluate_mitigation_citebench.py "
        "--config config.colab.yaml "
        "--dataset-role mitigation "
        f"--strategy {STRATEGY} "
        f"--context-source {CITEEVAL_CONTEXT_SOURCE} "
        "--system-source benchmark/CiteEval/data/system_eval/system_eval_examples.json "
        f"--max-samples {CITEEVAL_MAX_SAMPLES} "
        f"--provider {provider} "
        f"--model-name {model_name} "
        f"--variants {verifier_variants_arg}"
    )
    run(cmd, cwd=REPO_DIR)
else:
    print("RUN_CITEEVAL_VERIFIER_MODULE_EVAL=False, skipped.")

In [ ]:
# Export evaluation outputs manifest/snapshot
ensure_dir(DRIVE_OUTPUT_DIR)
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
export_root = Path(DRIVE_OUTPUT_DIR) / f"{RUN_TAG}_{STRATEGY}_{run_stamp}"
ensure_dir(export_root)

targets = [
    Path(REPO_DIR) / 'outputs' / 'ragtruth_eval',
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval',
    Path(REPO_DIR) / 'outputs' / 'mitigation_eval_citebench',
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'system_eval_outputs',
    Path(REPO_DIR) / 'benchmark' / 'CiteEval' / 'data' / 'metric_eval_outputs',
]

local_work_root_path = Path(LOCAL_WORK_ROOT).resolve()
artifacts_on_drive = str(local_work_root_path).startswith('/content/drive/')
exported_paths = []

if artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT:
    print("Artifacts are already on Drive; skipping duplicate copy.")
    for t in targets:
        if t.exists():
            exported_paths.append(str(t))
            print('Registered existing artifact:', t)
        else:
            print('Skip missing:', t)
else:
    for t in targets:
        if t.exists():
            dest = export_root / t.name
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(t, dest)
            exported_paths.append(str(dest))
            print('Exported:', t, '->', dest)
        else:
            print('Skip missing:', t)

manifest = {
    "timestamp": run_stamp,
    "strategy": STRATEGY,
    "local_work_root": LOCAL_WORK_ROOT,
    "repo_dir": REPO_DIR,
    "verifier_variants": VERIFIER_VARIANTS,
    "export_root": str(export_root),
    "targets": [str(t) for t in targets],
    "exported_artifacts": exported_paths,
    "artifacts_already_on_drive": artifacts_on_drive,
    "skipped_copy_when_persistent": bool(artifacts_on_drive and SKIP_EXPORT_COPY_WHEN_PERSISTENT),
}

manifest_path = export_root / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Done. Manifest:', manifest_path)